# 🧠 Intel Image Classification — TensorFlow / Keras CNN

**Dataset:** [Intel Image Classification](https://www.kaggle.com/datasets/puneet6060/intel-image-classification)  
**Framework:** TensorFlow / Keras  
**Output:** `your_firstname_model.keras` saved to Google Drive  

---

### 📋 Notebook Sections
1. GPU Check & Setup
2. Mount Google Drive
3. Install Dependencies
4. Download Dataset via Kaggle API
5. Data Pipeline (tf.data with Augmentation)
6. Model Architecture (IntelCNN)
7. Training with Callbacks
8. Evaluation & Plots
9. Save Model & Download

> ⚡ **Runtime:** Go to `Runtime → Change runtime type → T4 GPU` before running.

## 1️⃣ GPU Check & Setup

In [ ]:
import tensorflow as tf

# ── Configure GPU memory growth ────────────────────────────────
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"✅  {len(gpus)} GPU(s) detected")
    for i, gpu in enumerate(gpus):
        details = tf.config.experimental.get_device_details(gpu)
        print(f"    GPU {i}: {details.get('device_name', gpu.name)}")
else:
    print("⚠️  No GPU detected — running on CPU (will be slow).")
    print("   Go to Runtime → Change runtime type → T4 GPU")

print(f"\nTensorFlow version : {tf.__version__}")
print(f"Keras version      : {tf.keras.__version__}")
print(f"Devices available  : {[d.name for d in tf.config.list_logical_devices()]}")

## 2️⃣ Mount Google Drive

Models and checkpoints will be saved to `My Drive/intel_classifier/`.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_OUTPUT = '/content/drive/MyDrive/intel_classifier'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
print(f"📁  Output directory: {DRIVE_OUTPUT}")

## 3️⃣ Install Dependencies

In [ ]:
# TensorFlow is pre-installed on Colab — install kaggle API only
!pip install -q kaggle scikit-learn

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report
import itertools

print(f"numpy      : {np.__version__}")
print("✅  All dependencies ready")

## 4️⃣ Download Dataset via Kaggle API

### How to get your `kaggle.json`
1. Go to [kaggle.com](https://kaggle.com) → Profile → Settings → **Create New Token**
2. A `kaggle.json` downloads to your computer
3. Run the cell below — it will prompt you to upload that file

In [ ]:
from google.colab import files
import shutil

print("📤  Upload your kaggle.json file:")
uploaded = files.upload()

os.makedirs('/root/.kaggle', exist_ok=True)
shutil.move('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print("✅  kaggle.json configured")

In [ ]:
DATA_DIR = '/content/intel_data'
os.makedirs(DATA_DIR, exist_ok=True)

!kaggle datasets download -d puneet6060/intel-image-classification -p {DATA_DIR} --unzip

# ── Verify structure ───────────────────────────────────────────
for root, dirs, files_list in os.walk(DATA_DIR):
    depth = root.replace(DATA_DIR, '').count(os.sep)
    if depth < 3:
        indent = '  ' * depth
        print(f"{indent}{os.path.basename(root)}/")
        if depth == 2:
            n = len([f for f in files_list if f.lower().endswith(('.jpg','.jpeg','.png'))])
            print(f"{indent}  → {n} images")

## 5️⃣ Data Pipeline

**Augmentation strategy (train only):**
- Resize to 150×150 via `image_dataset_from_directory`
- RandomFlipLeftRight
- RandomBrightness / RandomContrast / RandomSaturation
- ImageNet normalisation (µ=[0.485,0.456,0.406], σ=[0.229,0.224,0.225])

**Validation split:** 10% stratified from training set

In [ ]:
from tensorflow import keras

# ── Hyperparameters ────────────────────────────────────────────
IMG_SIZE    = 150
BATCH_SIZE  = 64
CLASS_NAMES = ["buildings", "forest", "glacier", "mountain", "sea", "street"]
NUM_CLASSES = len(CLASS_NAMES)

# ── ImageNet normalisation constants ──────────────────────────
_MEAN = tf.constant([0.485, 0.456, 0.406], dtype=tf.float32)
_STD  = tf.constant([0.229, 0.224, 0.225], dtype=tf.float32)

def normalize(image, label):
    image = tf.cast(image, tf.float32) / 255.0
    image = (image - _MEAN) / _STD
    return image, label

def augment(image, label):
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_brightness(image, max_delta=0.3)
    image = tf.image.random_contrast(image, lower=0.7, upper=1.3)
    image = tf.image.random_saturation(image, lower=0.7, upper=1.3)
    image = tf.clip_by_value(image, 0.0, 255.0)  # keep in valid range before norm
    return image, label

# ── Locate folders ─────────────────────────────────────────────
def find_folder(root, candidates):
    for c in candidates:
        p = os.path.join(root, c)
        if os.path.isdir(p):
            return p
    raise FileNotFoundError(f"Dataset folder not found in {root}")

train_dir = find_folder(DATA_DIR, ["seg_train/seg_train", "seg_train"])
test_dir  = find_folder(DATA_DIR, ["seg_test/seg_test",  "seg_test"])

AUTOTUNE = tf.data.AUTOTUNE

# ── Training dataset ───────────────────────────────────────────
raw_train = keras.utils.image_dataset_from_directory(
    train_dir, labels='inferred', label_mode='int',
    image_size=(IMG_SIZE, IMG_SIZE), batch_size=None,
    shuffle=True, seed=42, validation_split=0.1, subset='training'
)
# ── Validation dataset ─────────────────────────────────────────
raw_val = keras.utils.image_dataset_from_directory(
    train_dir, labels='inferred', label_mode='int',
    image_size=(IMG_SIZE, IMG_SIZE), batch_size=None,
    shuffle=False, seed=42, validation_split=0.1, subset='validation'
)
# ── Test dataset ───────────────────────────────────────────────
raw_test = keras.utils.image_dataset_from_directory(
    test_dir, labels='inferred', label_mode='int',
    image_size=(IMG_SIZE, IMG_SIZE), batch_size=None, shuffle=False
)

train_ds = (raw_train.map(augment,    num_parallel_calls=AUTOTUNE)
                     .map(normalize,  num_parallel_calls=AUTOTUNE)
                     .batch(BATCH_SIZE).prefetch(AUTOTUNE))
val_ds   = (raw_val  .map(normalize,  num_parallel_calls=AUTOTUNE)
                     .batch(BATCH_SIZE).prefetch(AUTOTUNE))
test_ds  = (raw_test .map(normalize,  num_parallel_calls=AUTOTUNE)
                     .batch(BATCH_SIZE).prefetch(AUTOTUNE))

n_train = sum(1 for _ in raw_train)
n_val   = sum(1 for _ in raw_val)
n_test  = sum(1 for _ in raw_test)

print(f"Train : {n_train:,} samples")
print(f"Val   : {n_val:,} samples")
print(f"Test  : {n_test:,} samples")
print(f"Classes (inferred): {raw_train.class_names}")

In [ ]:
# ── Visualise a batch ──────────────────────────────────────────
MEAN_NP = np.array([0.485, 0.456, 0.406])
STD_NP  = np.array([0.229, 0.224, 0.225])

imgs_batch, labels_batch = next(iter(train_ds))
imgs_np   = imgs_batch.numpy()
labels_np = labels_batch.numpy()

fig, axes = plt.subplots(2, 8, figsize=(18, 5))
for i, ax in enumerate(axes.flat):
    if i >= 16: break
    img = (imgs_np[i] * STD_NP + MEAN_NP).clip(0, 1)
    ax.imshow(img)
    ax.set_title(CLASS_NAMES[labels_np[i]], fontsize=8)
    ax.axis('off')
plt.suptitle('Sample Training Images (after augmentation)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(DRIVE_OUTPUT, 'tf_sample_images.png'), dpi=120)
plt.show()
print("Sample grid saved to Drive.")

## 6️⃣ Model Architecture — IntelCNN

```
Input 150×150×3
 ├─ Stage 1: Conv(32) → Conv(64)  → MaxPool  → 75×75×64
 ├─ Stage 2: Conv(128)→ Conv(128) → MaxPool  → 37×37×128
 ├─ Stage 3: Conv(256)→ Conv(256) → MaxPool  → 18×18×256
 ├─ Stage 4: Conv(512)→ Conv(512) → MaxPool  →  9×9×512
 ├─ GlobalAveragePooling          → 512
 ├─ Dense(256, ReLU) + L2(1e-4)
 ├─ Dropout(0.4)
 └─ Dense(6, Softmax)
```
Each Conv block: **Conv2D → BatchNorm → ReLU**  
Weights: He Normal (conv), Glorot Uniform (dense)

In [ ]:
from tensorflow.keras import layers, regularizers

def conv_block(x, filters, pool=False, l2=1e-4):
    """Conv2D → BatchNorm → ReLU (→ optional MaxPool)."""
    x = layers.Conv2D(
        filters, (3, 3), padding='same', use_bias=False,
        kernel_regularizer=regularizers.l2(l2),
        kernel_initializer='he_normal'
    )(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    if pool:
        x = layers.MaxPooling2D((2, 2))(x)
    return x


def build_intel_cnn(img_size=150, num_classes=6):
    inputs = keras.Input(shape=(img_size, img_size, 3), name='input_image')

    x = conv_block(inputs, 32)
    x = conv_block(x, 64,  pool=True)   # Stage 1

    x = conv_block(x, 128)
    x = conv_block(x, 128, pool=True)   # Stage 2

    x = conv_block(x, 256)
    x = conv_block(x, 256, pool=True)   # Stage 3

    x = conv_block(x, 512)
    x = conv_block(x, 512, pool=True)   # Stage 4

    x = layers.GlobalAveragePooling2D(name='gap')(x)
    x = layers.Dense(256, activation='relu',
                     kernel_regularizer=regularizers.l2(1e-4),
                     kernel_initializer='glorot_uniform')(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(num_classes, activation='softmax',
                           name='predictions')(x)

    return keras.Model(inputs, outputs, name='IntelCNN')


# ── Build & inspect ────────────────────────────────────────────
model = build_intel_cnn(img_size=IMG_SIZE, num_classes=NUM_CLASSES)
model.summary(line_length=80)

total_params = model.count_params()
print(f"\nTotal parameters : {total_params:,}")

## 7️⃣ Training with Callbacks

| Setting | Value |
|---------|-------|
| Optimiser | AdamW (weight_decay=1e-4) |
| Loss | SparseCategoricalCrossentropy |
| LR schedule | CosineDecay (1e-3 → 1e-6) |
| EarlyStopping | patience=7 on val_accuracy |
| ReduceLROnPlateau | factor=0.5, patience=3 |
| ModelCheckpoint | best val_accuracy |

In [ ]:
from tensorflow.keras.callbacks import (
    ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, CSVLogger
)

# ── Config ─────────────────────────────────────────────────────
EPOCHS = 25
LR     = 1e-3

# ── Cosine decay LR schedule ───────────────────────────────────
steps_per_epoch = max(1, n_train // BATCH_SIZE)
total_steps     = steps_per_epoch * EPOCHS
lr_schedule = keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=LR,
    decay_steps=total_steps,
    alpha=1e-6,
)

# ── Compile ────────────────────────────────────────────────────
model.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=lr_schedule, weight_decay=1e-4),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=['accuracy'],
)

# ── Callbacks ──────────────────────────────────────────────────
CKPT_PATH   = os.path.join(DRIVE_OUTPUT, 'best_tf_checkpoint.keras')
CSV_PATH    = os.path.join(DRIVE_OUTPUT, 'tf_training_log.csv')

callbacks = [
    ModelCheckpoint(CKPT_PATH, monitor='val_accuracy',
                    save_best_only=True, verbose=1),
    EarlyStopping(monitor='val_accuracy', patience=7,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                      patience=3, min_lr=1e-7, verbose=1),
    CSVLogger(CSV_PATH),
]

print("✅  Model compiled")
print(f"   Steps/epoch : {steps_per_epoch}")
print(f"   Total steps : {total_steps}")
print(f"   Checkpoint  : {CKPT_PATH}")

In [ ]:
import time
t0 = time.time()

history = model.fit(
    train_ds,
    epochs=EPOCHS,
    validation_data=val_ds,
    callbacks=callbacks,
    verbose=1,
)

elapsed = time.time() - t0
best_val_acc = max(history.history['val_accuracy'])
print(f"\n⏱  Training time : {elapsed/60:.1f} min")
print(f"🏆  Best val acc : {best_val_acc*100:.2f}%")

## 8️⃣ Evaluation & Plots

In [ ]:
# ── Training curves ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['loss'],     label='Train', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Val',   linewidth=2, linestyle='--')
axes[0].set_title('Loss over Epochs', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['accuracy'],     label='Train', linewidth=2)
axes[1].plot(history.history['val_accuracy'], label='Val',   linewidth=2, linestyle='--')
axes[1].set_title('Accuracy over Epochs', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle('TensorFlow IntelCNN — Training History', fontsize=13, fontweight='bold')
plt.tight_layout()
plot_path = os.path.join(DRIVE_OUTPUT, 'tensorflow_training_history.png')
plt.savefig(plot_path, dpi=150)
plt.show()
print(f"Plot saved → {plot_path}")

In [ ]:
# ── Test set evaluation ────────────────────────────────────────
test_loss, test_acc = model.evaluate(test_ds, verbose=1)
print(f"\nTest Loss     : {test_loss:.4f}")
print(f"Test Accuracy : {test_acc*100:.2f}%")

In [ ]:
# ── Confusion matrix ───────────────────────────────────────────
all_preds, all_labels = [], []
for imgs, labels in test_ds:
    preds = model.predict(imgs, verbose=0)
    all_preds.extend(np.argmax(preds, axis=1))
    all_labels.extend(labels.numpy())

cm = confusion_matrix(all_labels, all_preds)
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.colorbar(im)
ax.set(xticks=range(NUM_CLASSES), yticks=range(NUM_CLASSES),
       xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
       xlabel='Predicted', ylabel='True',
       title='Confusion Matrix — Test Set')
plt.setp(ax.get_xticklabels(), rotation=30, ha='right')
thresh = cm.max() / 2
for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
    ax.text(j, i, cm[i, j], ha='center', va='center',
            color='white' if cm[i, j] > thresh else 'black', fontsize=11)
plt.tight_layout()
cm_path = os.path.join(DRIVE_OUTPUT, 'tensorflow_confusion_matrix.png')
plt.savefig(cm_path, dpi=150)
plt.show()
print(f"Confusion matrix saved → {cm_path}")

## 9️⃣ Save Model & Download

The model is saved in two ways:
- **Google Drive** → `My Drive/intel_classifier/your_firstname_model.keras`  
- **Direct download** → triggers a browser download for local use

In [ ]:
# ── Save final model to Google Drive ──────────────────────────
SAVE_PATH = os.path.join(DRIVE_OUTPUT, 'your_firstname_model.keras')

model.save(SAVE_PATH)

size_mb = os.path.getsize(SAVE_PATH) / 1e6
print(f"✅  Model saved to Drive → {SAVE_PATH}")
print(f"    File size : {size_mb:.1f} MB")
print(f"    Val  acc  : {best_val_acc*100:.2f}%")
print(f"    Test acc  : {test_acc*100:.2f}%")

# ── Also save training history as JSON ────────────────────────
import json
hist_path = os.path.join(DRIVE_OUTPUT, 'tf_history.json')
with open(hist_path, 'w') as f:
    json.dump(history.history, f)
print(f"    History   : {hist_path}")

In [ ]:
# ── Download to local machine ──────────────────────────────────
from google.colab import files
files.download(SAVE_PATH)
print("📥  Download started — save to your project's models/ folder")

## 🏠 Loading the Model Locally

After downloading `your_firstname_model.keras`, place it in your Flask project:

```
flask_app/
  models/
    your_firstname_model.keras   ← here
```

The Flask `app.py` loads it automatically. You can also load it manually:

```python
import tensorflow as tf
import numpy as np
from PIL import Image

# Load model
model = tf.keras.models.load_model('models/your_firstname_model.keras')

# Preprocess and predict
MEAN = np.array([0.485, 0.456, 0.406])
STD  = np.array([0.229, 0.224, 0.225])

img = Image.open('photo.jpg').convert('RGB').resize((150, 150))
arr = (np.array(img, dtype=np.float32) / 255.0 - MEAN) / STD
arr = np.expand_dims(arr, 0)  # (1, 150, 150, 3)

probs = model.predict(arr, verbose=0)[0]
CLASS_NAMES = ['buildings','forest','glacier','mountain','sea','street']
print('Predicted:', CLASS_NAMES[np.argmax(probs)],
      f'({probs.max()*100:.1f}%)')
```